In [3]:
###### LEITURA INCREMENTAL DE ARQUIVOS - processamento apenas das informações novas ######

#---------------------BIBLIOTECAS UTILIZADAS---------------------#
import time
from datetime import datetime
from file_read_backwards import FileReadBackwards

#-----------------------FUNÇÕES UTILIZADAS-----------------------#

#Para pegar uma informação em alguma linha (linha, posição da informação)
def dado_especifico(linha_desejada, n):
    lista_dados = linha_desejada.split(sep=",")
    lista_dados[0] = lista_dados[0].replace('"', "")
    informacao_desejada = lista_dados[n-1]
    return informacao_desejada

#Para substituir uma linha
def substituir_linha(nome_arquivo, numero_linha, novo_texto):
    # 1. Lê todas as linhas do arquivo
    with open(nome_arquivo, "r", encoding="utf-8") as arquivo:
        linhas = arquivo.readlines()

    # 2. Altera a linha desejada
    linhas[numero_linha] = novo_texto

    # 3. Reescreve o arquivo com a modificação
    with open(nome_arquivo, "w", encoding="utf-8") as arquivo:
        arquivo.writelines(linhas)

#Para colocar o ponteiro na linha anterior
def ler_linha_anterior(f):
    posicao_atual = f.tell()

    if posicao_atual == 0:
        return None

    # 1. Ignora o \n que terminou a linha atual
    posicao = posicao_atual - 1
    f.seek(posicao)

    if f.read(1) == b'\n':
        posicao -= 1

    # 2. Procura o \n que está antes da linha atual
    encontrou_fim_linha_anterior = False

    while posicao >= 0:
        f.seek(posicao)

        if f.read(1) == b'\n':
            encontrou_fim_linha_anterior = True
            posicao -= 1
            break

        posicao -= 1

    # Não existe linha anterior
    if not encontrou_fim_linha_anterior:
        return None

    # 3. Procura o \n que está antes da linha anterior
    while posicao >= 0:
        f.seek(posicao)

        if f.read(1) == b'\n':
            f.seek(posicao + 1)

            linha = f.readline()
            return linha.decode("utf-8")

        posicao -= 1

    # A linha anterior é a primeira linha do arquivo
    f.seek(0)

    linha = f.readline()
    return linha.decode("utf-8")

#-----------QUANTIDADE DE VEZES QUE O PROGRAMA IRÁ RODAR---------#

while True:
    try:
        quantidade_verificacoes = int(input("Digite a quantidade de vezes que o programa verificará se possui linhas novas: "))
        break
    except:
        continue

while True:
    try:
        tempo_cada_verificacao = int(input("Digite o tempo de pausa entre cada verificação (s): "))
        break
    except:
        continue

#----------------------------PRINCIPAL---------------------------#

#entrada de variáveis
records_total_lista = list()
nome_arquivo_dados_simulados = "/Users/kauad/OneDrive/Documentos/Universidade Federal de Minas Gerais/CEMTEC/Simulação da Estação Solarimétrica/dados_simulados.txt"

record_anterior = 0
with FileReadBackwards("posicao_atual.txt", encoding="utf-8") as frb:
    for linha_record in frb:
        record_anterior = int(linha_record[22:].strip())
        break

for i in range(quantidade_verificacoes):

    #----------------VERIFICAÇÃO DA ÚLTIMA POSIÇÃO---------------#

    file_posicao = open("posicao_atual.txt", "r", encoding="utf-8")
    linha_posicao = file_posicao.readline() #leitura da linha na qual informna a posição do arquivo
    ultima_posicao = linha_posicao[25:].strip() #.strip() remove os espaços em brancos
    ultima_posicao = int(ultima_posicao)
    file_posicao.close()

    #---------MANDA O PROGRAMA IR DIRETO PARA ESSA POSIÇÃO--------#
    file = open(nome_arquivo_dados_simulados, "rb")
    file.seek(ultima_posicao)

    #---------------------ENTRADA DE VARIÁVEIS---------------------#
    posicao_para_gravar = 0
    records_lista = list()

    #------------------LEITURA DE CADA LINHA NOVA------------------#
    while True:
        linha = file.readline()

        if linha == b"":
            posicao_para_gravar = file.tell()
            file.close()
            break

        linha = linha.decode("utf-8")

        linha_list = linha.split(',')

        #------------------------VERIFICAÇÕES----------------------#

        #Verificação se é o cabeçalho
        linha_lista = linha.split(',')
        lista_verificador_cabecalho = ["TOA5", "TIMESTAMP", "TS"]
        veririficador_cabecalho = linha_lista[0].replace('"',"")
        if veririficador_cabecalho in lista_verificador_cabecalho:
            continue

        #Verificação se chegou no final do arquivo
        if linha == '\n':
            continue

        #Verificação do Record
        record_verificacao = int(dado_especifico(linha, 2))
        diferenca_records = record_verificacao - record_anterior
        print(
            f"POSIÇÃO: {file.tell()} | "
            f"RECORD ANTERIOR: {record_anterior} | "
            f"RECORD ENCONTRADO: {record_verificacao} | "
            f"DIFERENÇA: {diferenca_records}"
        )

        if diferenca_records > 1:
            for j in range(diferenca_records):
                linha = ler_linha_anterior(file)

                if linha is None:
                    print("Não existe mais linha anterior.")
                    break

                print(
                    f"    VOLTANDO: {j+1} | "
                    f"RECORD: {dado_especifico(linha, 2)} | "
                    f"POSIÇÃO: {file.tell()}"
                )

        #-----------------PROCESSAMENTO DA LINHA NOVA---------------#
        record_linha_nova = int(dado_especifico(linha, 2))
        records_lista.append(record_linha_nova)

        record_anterior += 1

        if len(records_total_lista)!=0:
            substituir_linha("posicao_atual.txt", 1, f"record atual da linha: {records_total_lista[-1]}")

    #------------NÚMERO DE REGISTROS TOTAIS PROCESSADOS-------------#
    records_total_lista = records_total_lista + records_lista

    #--------------GRAVANDO A POSIÇÃO E O RECORD ATUAL--------------#
    substituir_linha("posicao_atual.txt", 0, f"posição atual do arquivo: {posicao_para_gravar}"+"\n")

    if len(records_total_lista)!=0:
        substituir_linha("posicao_atual.txt", 1, f"record atual da linha: {records_total_lista[-1]}")

    #---------------------FIM DE UMA VERIFICAÇÃO--------------------#
    print("-----------"*5)
    print(f"VERIFICAÇÃO DE NÚMERO {i+1} CONCLUIDA COM SUCESSO")
    print(f"Gravações processadas: {records_lista}")
    print(f"Quantidade de linhas novas: {len(records_lista)}")
    print("-----------"*5)
    print("\n")

    time.sleep(tempo_cada_verificacao)

#---------------------FIM DO PROGRAMA--------------------#
print("PROGRAMA FINALIZADO")
print(f"TOTAL DE GRAVAÇÕES PROCESSADAS: {records_total_lista}")
print(f"QUANTIDADE TOTAL DE LINHAS NOVAS: {len(records_total_lista)}")

IndexError: list index out of range